In [25]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
df = pd.read_excel('data.xlsx')
df.head()

,Company Code,Adviser Code,Adviser Name,Adviser Date of Birth,Account No.,Contract No.,Product Type,Fund House/Issuer/Exchange,Product Name,Product Category/DPMS Risk Level/Company Portfolio Risk Level,...,Account Type,Main Client NRIC,Main Client Name,Main Client Address,Joint Client NRIC,Joint Client Name,Pledged To,Created By,Unnamed: 24,Client number
0,NaN,NaN,NaN,NaN,G0004301,CON210901000158,UT,UBS Asset Management (Singapore) Ltd,UBS (Lux) Equity Fund - Greater China P Acc SGD,Equity,...,WP,NaN,NaN,NaN,NaN,NaN,-,Adviser,NaN,B2
1,NaN,NaN,NaN,NaN,G0004301,BDCON210901000026,DPMS,-,DPMS Capital Growth IGP (Aggressive),Aggressive,...,WP,NaN,NaN,NaN,NaN,NaN,-,Adviser,NaN,B2
2,NaN,NaN,NaN,NaN,G0024923,BSTK20210902000093,STOCK,NASDAQ,Lucid Group,Equity,...,WP,NaN,NaN,NaN,NaN,NaN,-,-,NaN,B3
3,NaN,NaN,NaN,NaN,G0024923,BSTK20210902000091,STOCK,NASDAQ,Lucid Group,Equity,...,WP,NaN,NaN,NaN,NaN,NaN,-,-,NaN,B3
4,NaN,NaN,NaN,NaN,G0024923,BSTK20210902000092,STOCK,NYSE,ChargePoint Holdings,Equity,...,WP,NaN,NaN,NaN,NaN,NaN,-,-,NaN,B3


In [4]:
df.columns

Index(['Company Code', 'Adviser Code', 'Adviser Name', 'Adviser Date of Birth',
       'Account No.', 'Contract No.', 'Product Type',
       'Fund House/Issuer/Exchange', 'Product Name',
       'Product Category/DPMS Risk Level/Company Portfolio Risk Level',
       'Payment Method', 'Transaction Type', 'Transaction Mode',
       'Transaction Date', 'Remark', 'Transaction Amount (SGD)',
       'Account Type', 'Main Client NRIC', 'Main Client Name',
       'Main Client Address', 'Joint Client NRIC', 'Joint Client Name',
       'Pledged To', 'Created By', 'Unnamed: 24', 'Client number'],
      dtype='object')

In [5]:
df.shape

(3171, 26)

In [6]:
df.isnull().sum()

Company Code                                                     3171
Adviser Code                                                     3171
Adviser Name                                                     3171
Adviser Date of Birth                                            3171
Account No.                                                         0
Contract No.                                                        0
Product Type                                                        0
Fund House/Issuer/Exchange                                          0
Product Name                                                        0
Product Category/DPMS Risk Level/Company Portfolio Risk Level       0
Payment Method                                                      0
Transaction Type                                                    0
Transaction Mode                                                    0
Transaction Date                                                    0
Remark              

In [7]:
df = df.drop(['Company Code','Adviser Code','Adviser Name','Adviser Date of Birth','Main Client NRIC','Main Client Name',
       'Main Client Address', 'Joint Client NRIC', 'Joint Client Name','Unnamed: 24'],axis=1)

In [8]:
df.shape

(3171, 16)

In [9]:
df.isnull().sum()

Account No.                                                      0
Contract No.                                                     0
Product Type                                                     0
Fund House/Issuer/Exchange                                       0
Product Name                                                     0
Product Category/DPMS Risk Level/Company Portfolio Risk Level    0
Payment Method                                                   0
Transaction Type                                                 0
Transaction Mode                                                 0
Transaction Date                                                 0
Remark                                                           0
Transaction Amount (SGD)                                         0
Account Type                                                     0
Pledged To                                                       0
Created By                                                    

In [10]:
df.nunique()   

Account No.                                                       223
Contract No.                                                     3171
Product Type                                                        5
Fund House/Issuer/Exchange                                        100
Product Name                                                      566
Product Category/DPMS Risk Level/Company Portfolio Risk Level       9
Payment Method                                                      4
Transaction Type                                                    1
Transaction Mode                                                    2
Transaction Date                                                  863
Remark                                                              1
Transaction Amount (SGD)                                         2691
Account Type                                                        6
Pledged To                                                          3
Created By          

In [11]:
df = df.drop(['Remark','Transaction Type'],axis=1)

In [12]:
df.columns

Index(['Account No.', 'Contract No.', 'Product Type',
       'Fund House/Issuer/Exchange', 'Product Name',
       'Product Category/DPMS Risk Level/Company Portfolio Risk Level',
       'Payment Method', 'Transaction Mode', 'Transaction Date',
       'Transaction Amount (SGD)', 'Account Type', 'Pledged To', 'Created By',
       'Client number'],
      dtype='object')

In [13]:
# df.to_excel('cleaned_data.xlsx',index=False)

In [18]:
df_rec = df[['Client number', 'Product Type']].copy()

# df_rec['Product Type'].unique()

print("Unique clients:", df_rec['Client number'].nunique())
print("Unique product types:", df_rec['Product Type'].nunique())

df_rec.head()

Unique clients: 190
Unique product types: 5


,Client number,Product Type
0,B2,UT
1,B2,DPMS
2,B3,STOCK
3,B3,STOCK
4,B3,STOCK


In [23]:
basket = (
    df.groupby(['Client number'])['Product Type']
    .unique()
    .reset_index()
)
basket

,Client number,Product Type
0,B1,"[DPMS, STOCK, ETF]"
1,B10,"[STOCK, ETF, BOND]"
2,B100,"[ETF, DPMS]"
3,B101,[ETF]
4,B102,[DPMS]
...,...,...
185,B95,"[UT, STOCK, BOND]"
186,B96,[DPMS]
187,B97,[ETF]
188,B98,"[ETF, STOCK]"


In [24]:
te = TransactionEncoder()
te_ary = te.fit(basket['Product Type']).transform(basket['Product Type'])
basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)

basket_encoded.head()

,BOND,DPMS,ETF,STOCK,UT
0,False,True,True,True,False
1,True,False,True,True,False
2,False,True,True,False,False
3,False,False,True,False,False
4,False,True,False,False,False


In [26]:
# Cell 4: Find frequent itemsets using Apriori
frequent_itemsets = apriori(basket_encoded, min_support=0.1, use_colnames=True)
frequent_itemsets.sort_values(by='support', ascending=False, inplace=True)

print("Frequent Itemsets:")
display(frequent_itemsets.head(10))


Frequent Itemsets:


,support,itemsets
3,0.478947,(STOCK)
1,0.468421,(DPMS)
2,0.394737,(ETF)
4,0.342105,(UT)
9,0.289474,"(ETF, STOCK)"
6,0.194737,"(ETF, DPMS)"
0,0.184211,(BOND)
7,0.157895,"(STOCK, DPMS)"
8,0.152632,"(DPMS, UT)"
12,0.142105,"(STOCK, ETF, DPMS)"


In [27]:
# Cell 5: Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules.sort_values(by=['lift', 'confidence'], ascending=[False, False], inplace=True)

# Keep only useful columns
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

# Convert frozensets to strings
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

display(rules.head(10))

,antecedents,consequents,support,confidence,lift
8,ETF,"STOCK, DPMS",0.142105,0.360000,2.280000
5,"STOCK, DPMS",ETF,0.142105,0.900000,2.280000
0,ETF,STOCK,0.289474,0.733333,1.531136
1,STOCK,ETF,0.289474,0.604396,1.531136
7,STOCK,"ETF, DPMS",0.142105,0.296703,1.523612
6,"ETF, DPMS",STOCK,0.142105,0.729730,1.523612
11,BOND,DPMS,0.100000,0.542857,1.158909
10,DPMS,BOND,0.100000,0.213483,1.158909
2,ETF,DPMS,0.194737,0.493333,1.053184
3,DPMS,ETF,0.194737,0.415730,1.053184


In [28]:
# Cell 6: Generate recommendations per client based on rules

def recommend_for_client(client_products, rules_df, top_n=3):
    client_products = set(client_products)
    recommendations = []
    
    for _, row in rules_df.iterrows():
        antecedent = set(row['antecedents'].split(', '))
        consequent = set(row['consequents'].split(', '))
        
        if antecedent.issubset(client_products):
            new_recos = consequent - client_products
            for rec in new_recos:
                recommendations.append((rec, row['confidence'], row['lift']))
    
    # Sort by lift and confidence
    recos_df = pd.DataFrame(recommendations, columns=['Product Type', 'Confidence', 'Lift'])
    recos_df = recos_df.sort_values(by=['Lift', 'Confidence'], ascending=False).drop_duplicates('Product Type')
    return recos_df.head(top_n)['Product Type'].tolist()

# Apply for each client
recommendations = []
for _, row in basket.iterrows():
    recs = recommend_for_client(row['Product Type'], rules)
    recommendations.append({
        'Client number': row['Client number'],
        'Bought Product Types': ', '.join(row['Product Type']),
        'Recommended Product Types': ', '.join(recs) if recs else 'No Recommendation'
    })

reco_df = pd.DataFrame(recommendations)
display(reco_df.head(10))


,Client number,Bought Product Types,Recommended Product Types
0,B1,"DPMS, STOCK, ETF",BOND
1,B10,"STOCK, ETF, BOND",DPMS
2,B100,"ETF, DPMS","STOCK, BOND"
3,B101,ETF,"STOCK, DPMS"
4,B102,DPMS,"BOND, ETF, STOCK"
5,B103,DPMS,"BOND, ETF, STOCK"
6,B104,BOND,DPMS
7,B105,STOCK,"ETF, DPMS"
8,B106,STOCK,"ETF, DPMS"
9,B107,DPMS,"BOND, ETF, STOCK"
